# 19 — Performance, callbacks, and code generation

**Objectives.** Measure graph construction separately from evaluation, use mapped evaluation, generate C, and understand extension boundaries. Optimize only after profiling representative dimensions. SX often evaluates small scalar formulas efficiently; MX prevents large matrix algorithms from being scalar-expanded.


In [ ]:
import casadi as ca
import os
import pathlib
x=ca.MX.sym('x',4); p=ca.MX.sym('p',4)
f=ca.Function('generated_dynamics',[x,p],[ca.sin(x)+p*x**2,ca.jacobian(ca.sin(x)+p*x**2,x)])
mapped=f.map(100)
result=mapped(ca.DM.zeros(4,100),ca.DM.ones(4,100))
print('mapped output shape',result[0].shape,'nodes',f.n_nodes())
output=pathlib.Path('generated').resolve(); output.mkdir(exist_ok=True)
original=pathlib.Path.cwd()
try:
    os.chdir(output); f.generate('generated_dynamics.c',{'with_header':True})
finally:
    os.chdir(original)
assert (output/'generated_dynamics.c').exists()


In [ ]:
class SquareCallback(ca.Callback):
    def __init__(self,name='square_callback'):
        ca.Callback.__init__(self); self.construct(name,{'enable_fd':True})
    def get_n_in(self): return 1
    def get_n_out(self): return 1
    def eval(self,args): return [args[0]**2]
callback=SquareCallback(); z=ca.MX.sym('z')
wrapped=ca.Function('wrapped',[z],[callback(z)])
print('callback value',wrapped(3),'finite-difference derivative',ca.Function('dwrapped',[z],[ca.jacobian(wrapped(z),z)])(3))


Generated C removes virtual-machine overhead and can run without CasADi when linked against its self-contained API. `external` loads compatible compiled functions; `Function.jit` compiles C source at runtime. `Callback` connects Python/external logic but Python callbacks are slower and need supplied derivatives or finite differences.

**Common mistakes:** timing graph construction as evaluation, generating one enormous expanded function, and assuming threads make a non-thread-safe callback safe. **Recap:** establish graph boundaries, map repeated work, record timings, then compare VM, expanded, and generated implementations.
